In [1]:
from datasets import load_dataset, DatasetDict, concatenate_datasets
import os

In [2]:
dataset_names = [
    "TAIDE-EDU/task4_training_data_article_input",
    "TAIDE-EDU/task4_training_data_conversation_input",
    "TAIDE-EDU/reading_training_ground_truth_data_1107",
    "TAIDE-EDU/cloze_training_ground_truth_data_1107"
]

In [3]:
source_columns = [
    '閱讀測驗(含前置文章)',
    '閱讀測驗(不含前置文章)',
    '選詞填空(含前置文章)',
    '選詞填空(不含前置文章)'
]

In [4]:
import json
import copy
def flatten_and_format(batch):
    """
    將指定欄位(source_columns) 中的內容展開。
    1. 注入 Special Token 到 messages[0]['content']。
    2. 將 meta 轉為 JSON string。
    3. 過濾掉空的 messages 以確保訓練穩定性。
    """
    new_messages = []
    new_metadata = [] # [修改] 變數名稱改為 new_metadata
    
    for col in source_columns:
        # 防呆：如果該資料集根本沒有這個欄位，就跳過
        if col not in batch:
            continue
        
        # --- [功能新增] 判斷 Special Token ---
        # 建議採 2 種 Token 分類法，讓模型自己學會看有無 context
        prefix_token = ""
        if '閱讀測驗' in col or 'reading' in col:
            prefix_token = "<|task-4_1-indicator|>" # 閱讀測驗 Token
        elif '選詞填空' in col or 'cloze' in col:
            prefix_token = "<|task-4_2-indicator|>" # 選詞填空 Token
        else:
            prefix_token = "" # 若無匹配則不加

        for val in batch[col]:
            # 1. 檢查 val 是否為有效的字典
            if val is None or not isinstance(val, dict):
                continue
            
            # 2. 取得 messages
            msgs = val.get('messages')
            
            # 3. 取得 meta (若無則給空字典)
            # 假設原始資料裡的 key 還是 'meta'，讀取後我們存入 metadata 欄位
            raw_meta = val.get('meta', {})
            
            # 4. 判斷內容並加入
            # [重要修正] 
            # 只有當 msgs 有內容時才處理。
            # 如果 msgs 是空列表 []，我們直接跳過 (continue)，不要 append 空列表。
            # 這樣可以避免訓練時 Data Collator 報錯或產生無效 Loss。
            if msgs and len(msgs) > 0:
                
                # --- [功能新增] 注入 Special Token ---
                # 使用 deepcopy 避免修改原始資料參照
                current_msgs = copy.deepcopy(msgs)
                
                # 確保第一則訊息有 content 欄位 (防呆)
                if 'content' in current_msgs[0]:
                    original_content = current_msgs[0]['content']
                    # 將 Token 加在最前面
                    current_msgs[0]['content'] = prefix_token + original_content
                
                # --- meta 處理 ---
                # 將 meta 字典轉成 JSON String，避免欄位結構不一致
                meta_str = json.dumps(raw_meta, ensure_ascii=False)
                
                # 同步 append 到兩個列表
                new_messages.append(current_msgs)
                new_metadata.append(meta_str) # [修改] append 到 new_metadata
            
            else:
                # 若 messages 為空，直接跳過這筆資料
                # 確保 new_messages 和 new_metadata 的長度始終保持一致
                new_messages.append([])
                new_metadata.append(meta_str)

    # [修改] 回傳的 key 改為 "metadata"
    return {"messages": new_messages, "metadata": new_metadata}

# 用來存放處理好的 Dataset 物件processed_datasets = []
processed_datasets = []
print(f"準備處理 {len(dataset_names)} 個資料集...")

for name in dataset_names:
    try:
        print(f"正在載入: {name}")
        ds = load_dataset(name, split="train")
        
        print(f" - 原始筆數: {len(ds)}")
        
        # 執行 map 轉換
        processed_ds = ds.map(
            flatten_and_format,
            batched=True,
            remove_columns=ds.column_names,
            desc=f"Processing {name}"
        )
        
        print(f" - 轉換後筆數: {len(processed_ds)}")
        processed_datasets.append(processed_ds)
        
    except Exception as e:
        print(f"處理資料集 {name} 時發生錯誤: {e}")

if processed_datasets:
    print("-" * 30)
    print("正在合併所有資料集...")
    
    full_dataset_train = concatenate_datasets(processed_datasets)
    
    # 打亂資料
    full_dataset_train = full_dataset_train.shuffle(seed=42)
    
    print(f"合併完成！總資料筆數: {len(full_dataset_train)}")
    
    # 包裝成 DatasetDict
    final_dataset_dict = DatasetDict({
        "train": full_dataset_train
    })

    # 檢查第一筆資料 (確認 Token 是否成功加入)
    if len(final_dataset_dict['train']) > 0:
        print("範例資料 (第一筆):")
        first_msg = final_dataset_dict['train'][0]['messages']
        print("Messages Content[0]:", first_msg[0]['content'][:100] + "...") # 只印前100字預覽
        print("Metadata (str):", final_dataset_dict['train'][0]['metadata']) 
    
    # 儲存
    save_path = "./processed_all_data"
    final_dataset_dict.save_to_disk(save_path) 
    print(f"資料已儲存至: {save_path}")

else:
    print("沒有任何資料集被成功處理。")

準備處理 4 個資料集...
正在載入: TAIDE-EDU/task4_training_data_article_input


Generating train split: 100%|██████████| 1453/1453 [00:00<00:00, 26943.86 examples/s]


 - 原始筆數: 1453


Processing TAIDE-EDU/task4_training_data_article_input: 100%|██████████| 1453/1453 [00:00<00:00, 1695.46 examples/s]


 - 轉換後筆數: 5812
正在載入: TAIDE-EDU/task4_training_data_conversation_input


...

 - 原始筆數: 4


Processing TAIDE-EDU/cloze_training_ground_truth_data_1107: 100%|██████████| 4/4 [00:00<00:00, 501.50 examples/s]


 - 轉換後筆數: 4
------------------------------
正在合併所有資料集...
合併完成！總資料筆數: 11695
範例資料 (第一筆):
Messages Content[0]: <|task-4_2-indicator|>請撰寫一篇選詞填空「TOCFL 流利精通」級裡的【挖空後的文章】。...
Metadata (str): {"fix_perplexity": 0.9255653501838086, "generate_article_perplexity": 0.6000132130109086, "generate_questions_perplexity": 0.9204185123350295, "前置課文資訊": {"content": "張士豪和李雅婷是大學同學，他們常常一起討論社會問題。\n張士豪：雅婷，你有沒有發現，現代社會中的年輕人覺得工作壓力很大？\n李雅婷：是啊，我父母那一代，他們的時代比較穩定，可是現在的年輕人要適應很多變化。\n張士豪：對，現在的科技進步很快，像手機和網際網路的發展，改變了我們的日常生活。\n李雅婷：沒錯，我覺得這是一個世代之間的變化。老一輩習慣的是面對面的溝通，而年輕人更喜歡用社交媒體。\n張士豪：我還是覺得面對面的交流比較真實，像是朋友聚會這樣的情況下，大家能更好地互動。\n李雅婷：我也這麼想。不過，隨著社會的發展，年輕人也開始學習如何利用科技來提高效率。\n張士豪：是的，這是很好的趨勢。我們可以學習老一輩的經驗，同時也要用科技來解決問題。\n李雅婷：這樣的合作，也許能讓社會變得更好，也能讓我們面對未來的挑戰。\n張士豪：沒錯，我們需要找到平衡，讓不同世代之間更好地合作，共同創造更美好的社會。", "level": "第四級", "subtopic": "世代正義", "title": "社會變遷與未來", "topic": "社會"}, "完整文章": "社會發展的腳步日益加快，世代正義逐漸成為公共討論的焦點。年輕人常感受到窮困與不安，認為資源分配不均，機會受到上一代的影響而受限。固然長輩累積了豐富的經驗，卻未必能完全理解新世代所面臨的挑戰。相對地，年輕人不但仰賴科技來改善生活品質，還積極尋求創新方式以突破現狀。這種鬥智的局面，既是進步的動力，也是世代間矛盾的根源。既然社會變遷接二

Saving the dataset (1/1 shards): 100%|██████████| 11695/11695 [00:00<00:00, 155613.18 examples/s]

資料已儲存至: ./processed_all_data


In [5]:
print(final_dataset_dict.column_names)

{'train': ['messages', 'metadata']}


In [6]:
for i in range(20):
    if len(final_dataset_dict) > 0:
        print("範例資料:")
        print(final_dataset_dict['train'][i]['messages'])

範例資料:
[{'content': '<|task-4_2-indicator|>請撰寫一篇選詞填空「TOCFL 流利精通」級裡的【挖空後的文章】。', 'role': 'user'}, {'content': '【挖空後的文章】\n社會發展的腳步日益加快，世代正義逐漸成為公共討論的焦點。年輕人常感受到__1__與不安，認為資源分配不均，機會受到上一代的影響而受限。__2__長輩累積了豐富的經驗，__2__未必能完全理解新世代所面臨的挑戰。__3__，年輕人不但__4__科技來改善生活品質，還積極尋求創新方式以突破現狀。這種__5__的局面，既是進步的動力，也是世代間矛盾的根源。既然社會變遷接二連三地發生，大家就__6__地必須斟酌彼此的觀點，才能蔚為公正且永續的未來。', 'role': 'assistant'}, {'content': '請根據以上【挖空後的文章】出數題選擇題，每題需提供題號、四個選項、其中只有一個正確答案可填入文章後語句通順，將數題選擇題放至【題目】下方，數題參考答案放至【參考答案】後方。', 'role': 'user'}, {'content': '【題目】\n1.\n(A) 墊子\n(B) 安逸\n(C) 威風凜凜\n(D) 窮困\n\n2.\n(A) 一⋯⋯就⋯⋯\n(B) 不但⋯⋯還⋯⋯\n(C) 寧願⋯⋯也不⋯⋯\n(D) 固然⋯⋯卻⋯⋯\n\n3.\n(A) 按照\n(B) 至\n(C) 相對地\n(D) 離\n\n4.\n(A) 吹噓\n(B) 仰賴\n(C) 遺棄\n(D) 探視\n\n5.\n(A) 鬥智\n(B) 墊子\n(C) 鯉魚\n(D) 低階\n\n6.\n(A) 得宜\n(B) 無精打采\n(C) 責無旁貸\n(D) 平復\n\n\n【參考答案】:D、D、C、B、A、C', 'role': 'assistant'}, {'content': '請撰寫每題答案的參考解析，將數題參考解析依題號放至【參考解析】下方。', 'role': 'user'}, {'content': '【參考解析】\n1. 『窮困』指資源匱乏，與『不安』並列，最符合語境。其他選項語意不合或搭配不當。\n2. 『固然⋯⋯卻⋯⋯』強調前後對比，肯定長輩經驗，指出理解上的不足。其他選項語法或語意不合。\n3. 『相對地』用於前後行為對比，語意

## Task 123

In [7]:
from datasets import load_dataset

In [8]:
new_dataset_id = "TAIDE-EDU/Edu-TAIDE-IT-Data"
new_data_config = "task-1-2-3-241127"

print(f"\n正在下載/載入新資料: {new_dataset_id} ({new_data_config}) ...")

try:
    new_ds = load_dataset(new_dataset_id, new_data_config)
    new_train = new_ds['train']
    
    print(f" ✅ 新資料載入成功: {len(new_train)} 筆")
    print(f"    欄位: {new_train.column_names}")
    
except Exception as e:
    print(f" ❌ 新資料載入失敗: {e}")


正在下載/載入新資料: TAIDE-EDU/Edu-TAIDE-IT-Data (task-1-2-3-241127) ...
 ✅ 新資料載入成功: 146029 筆
    欄位: ['messages', 'source', 'metadata']


In [9]:
from datasets import load_from_disk
old_data_path = "./processed_all_data"
print(f"正在載入舊資料: {old_data_path} ...")

try:
    # 假設你的舊資料儲存格式是 DatasetDict
    old_ds_dict = load_from_disk(old_data_path)
    # 取得 train split
    old_train = old_ds_dict['train']
    
    print(f" ✅ 舊資料載入成功: {len(old_train)} 筆")
    print(f"    欄位: {old_train.column_names}")
    
except Exception as e:
    print(f" ❌ 舊資料載入失敗: {e}")

正在載入舊資料: ./processed_all_data ...
 ✅ 舊資料載入成功: 11695 筆
    欄位: ['messages', 'metadata']


In [10]:
def align_format(example):
        """
        將新資料格式轉換為舊資料格式：
        1. 取出 metadata (若無則為空字典)
        2. 將 metadata 字典轉為 JSON 字串
        3. 直接忽略/丟棄 source 欄位
        """
        # 1. 取得 metadata，確保是字典
        raw_meta = example.get('metadata')
        if raw_meta is None:
            raw_meta = {}
        
        # [修改] 這裡原本會將 source 併入 metadata，現在已移除。
        # 也就是說 source 資訊會在這裡被丟棄。
        
        # 2. 轉成 JSON String (與舊資料格式對齊)
        meta_str = json.dumps(raw_meta, ensure_ascii=False)
        
        # 3. 回傳
        return {
            'messages': example['messages'],
            'metadata': meta_str
        }

In [11]:
print("\n正在對齊新資料格式 (Drop source)...")

# remove_columns 很重要，它會移除原本不符合格式的 'source' 和舊 'metadata'
# 確保轉換後的 Dataset 只有我們 return 的那些欄位
new_train_formatted = new_train.map(
    align_format,
    remove_columns=new_train.column_names,
    desc="Aligning Schema"
)

print(f" ✅ 轉換完成！新資料欄位已變更為: {new_train_formatted.column_names}")


正在對齊新資料格式 (Drop source)...


Aligning Schema: 100%|██████████| 146029/146029 [00:03<00:00, 43324.43 examples/s]

 ✅ 轉換完成！新資料欄位已變更為: ['messages', 'metadata']


In [12]:
if set(old_train.column_names) == set(new_train_formatted.column_names):
    print("\n欄位一致，開始合併...")
    
    # 合併 list 中的 dataset
    combined_dataset = concatenate_datasets([old_train, new_train_formatted])
    
    # 打亂順序 (Shuffle)
    combined_dataset = combined_dataset.shuffle(seed=42)
    
    print(f" ✅ 合併完成！")
    print(f"    - 總筆數: {len(combined_dataset)}")
    
    # ==========================================
    # 5. 檢查結果範例
    # ==========================================
    print("\n[檢查合併後的第一筆資料]:")
    sample = combined_dataset[0]
    print("Messages (預覽):", str(sample['messages'])[:100] + "...")
    print("Metadata (JSON str):", sample['metadata'])
    
    # ==========================================
    # 6. 儲存
    # ==========================================
    save_path = "./final_merged_dataset"
    
    # 包裝回 DatasetDict
    final_output = DatasetDict({
        'train': combined_dataset
    })
    
    print(f"\n正在儲存至: {save_path} ...")
    final_output.save_to_disk(save_path)
    print(" 🎉 所有作業完成！")
    
else:
    print("\n ❌ 欄位不一致，無法合併！")
    print(f"    舊資料欄位: {old_train.column_names}")
    print(f"    新資料欄位: {new_train_formatted.column_names}")


欄位一致，開始合併...
 ✅ 合併完成！
    - 總筆數: 157724

[檢查合併後的第一筆資料]:
Messages (預覽): [{'content': '<|task-2-indicator|>請判斷「任事」在以下句子中為何種解釋，並直接輸出正確的選項代號。\n《金瓶梅》第七二回：「名永壽，貼刑，不止二十歲，捏出水兒來的一個...
Metadata (JSON str): "{\"ans\": \"無論什麼事。\", \"classical\": 1, \"explanations\": [\"委任職務。\", \"擔當事務。\", \"無論什麼事。\"], \"position\": [36, 38], \"sentence\": \"《金瓶梅》第七二回：「名永壽，貼刑，不止二十歲，捏出水兒來的一個小後生，任事兒不知道。」\", \"word\": \"任事\"}"

正在儲存至: ./final_merged_dataset ...


Saving the dataset (0/1 shards):   0%|          | 0/157724 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 157724/157724 [00:00<00:00, 334766.47 examples/s]

 🎉 所有作業完成！


## 確認資料

In [13]:
from datasets import load_from_disk

dataset_path = "/home/chris/LLM-Training/final_merged_dataset"

# 2. 讀取資料
print(f"正在讀取資料: {dataset_path} ")
dataset = load_from_disk(dataset_path)

# 3. 檢查資料結構
print("\n=== 資料集資訊 ===")
print(dataset)

正在讀取資料: /home/chris/LLM-Training/final_merged_dataset 

=== 資料集資訊 ===
DatasetDict({
    train: Dataset({
        features: ['messages', 'metadata'],
        num_rows: 157724
    })
})


In [14]:
print("\n=== 檢查前 10 筆資料的長度 (確認 Shuffle 狀況) ===")
for i in range(10):
    # 假設你的資料裡面有 'messages' 或 'input_ids' 欄位
    # 這裡先抓 messages 看長度，如果已經 tokenize 過了就抓 input_ids
    if 'messages' in dataset.column_names:
        content_len = len(str(dataset[i]['messages']))
        print(f"Index {i}: 長度約 {content_len}")
    elif 'input_ids' in dataset.column_names:
        content_len = len(dataset[i]['input_ids'])
        print(f"Index {i}: Token 長度 {content_len}")
    else:
        print(f"Index {i}: {dataset['train'][i]}")


=== 檢查前 10 筆資料的長度 (確認 Shuffle 狀況) ===
Index 0: {'messages': [{'content': '<|task-2-indicator|>請判斷「任事」在以下句子中為何種解釋，並直接輸出正確的選項代號。\n《金瓶梅》第七二回：「名永壽，貼刑，不止二十歲，捏出水兒來的一個小後生，<|task-2-bow|>任事<|task-2-eow|>兒不知道。」\n\nA. 委任職務。\nB. 擔當事務。\nC. 無論什麼事。', 'role': 'user'}, {'content': 'C', 'role': 'assistant'}], 'metadata': '"{\\"ans\\": \\"無論什麼事。\\", \\"classical\\": 1, \\"explanations\\": [\\"委任職務。\\", \\"擔當事務。\\", \\"無論什麼事。\\"], \\"position\\": [36, 38], \\"sentence\\": \\"《金瓶梅》第七二回：「名永壽，貼刑，不止二十歲，捏出水兒來的一個小後生，任事兒不知道。」\\", \\"word\\": \\"任事\\"}"'}
Index 1: {'messages': [{'content': '<|task-1-indicator|>創作一篇TBCL 5 級的華語短文課文，以「情緒、態度」為核心。請從目標設定以及人際互動以及潛能開發以及信任建立以及自我認識等子主題中，選擇一至多個可互相呼應的元素，編寫一篇結構完整、內容連貫的課文。', 'role': 'user'}, {'content': '了解自我與潛能開發\n\n在生活中，了解自我並開發潛能是達成個人目標的重要步驟。每個人都有獨特的優勢與潛力，但要如何發掘並善用這些潛力呢？\n\n首先，我們需要清楚自己的長處與短處。透過自我評估，我們可以更清楚地看到哪些方面需要加強，哪些領域我們已有足夠的能力。這樣一來，我們能夠針對自己的優勢制定更具體的目標。\n\n接下來，在設定目標的過程中，我們應該確保這些目標是具體、可衡量、可達成、相關且有時間限制的。這樣的目標設定能夠幫助我們有計畫地逐步進步，並在完成小目標後獲得成就感。\n\n再者，人際互動也是開發潛能的重要一環。與他人分享你的目標與挑

## Add Special Token

In [2]:
from transformers import AutoTokenizer

# 1. 設定原始路徑 (跟您 YAML 裡的一樣)
base_model_path = "TAIDE-EDU/phi-3.5-mini-instruct_zhtw_ld1_hq3.1_b8.3-p3_st-task-1-2-3-v3_e4-s2525"

# 2. 設定您要新增的 Tokens
new_tokens = ["<|task-4_1-indicator|>", "<|task-4_2-indicator|>"]

# 3. 載入原始 Tokenizer
print(f"正在載入: {base_model_path}")
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

# 4. 新增 Tokens
num_added_toks = tokenizer.add_tokens(new_tokens)
print(f"成功新增了 {num_added_toks} 個 Token")

# 5. 儲存到本地資料夾 (例如存到 ./custom_tokenizer)
save_path = "/home/u5534225/LLM-Training/src/llm_training/tokenizer"
tokenizer.save_pretrained(save_path)
print(f"新 Tokenizer 已儲存至: {save_path}")

# 驗證一下
print("驗證 task-4_1-indicator ID:", tokenizer.convert_tokens_to_ids("<|task-4_1-indicator|>"))
print("驗證 task-4_2-indicator ID:", tokenizer.convert_tokens_to_ids("<|task-4_2-indicator|>"))

正在載入: TAIDE-EDU/phi-3.5-mini-instruct_zhtw_ld1_hq3.1_b8.3-p3_st-task-1-2-3-v3_e4-s2525
成功新增了 2 個 Token
新 Tokenizer 已儲存至: /home/u5534225/LLM-Training/src/llm_training/tokenizer
驗證 task-4_1-indicator ID: 97717
驗證 task-4_2-indicator ID: 97718


## Mistral add special token

In [1]:
from transformers import PreTrainedTokenizerFast
from huggingface_hub import hf_hub_download

# 1️⃣ 載入官方 tokenizer
repo_id = "mistralai/Ministral-3-14B-Instruct-2512-BF16"
tokenizer_file = hf_hub_download(repo_id=repo_id, filename="tokenizer.json")
tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_file)

# 2️⃣ 設定基本 special tokens（與官方一致）
tokenizer.bos_token = "<s>"
tokenizer.eos_token = "</s>"
tokenizer.unk_token = "<unk>"
tokenizer.pad_token = "<pad>"

mistral_chat_template = """{#- Begin of sequence token. #}
{{- bos_token }}
{#- Handle system prompt if it exists. #}
{%- if messages[0]['role'] == 'system' %}
    {{- '[SYSTEM_PROMPT]' -}}
    {%- if messages[0]['content'] is string %}
        {{- messages[0]['content'] -}}
    {%- else %}        
        {%- for block in messages[0]['content'] %}
            {%- if block['type'] == 'text' %}
                {{- block['text'] }}
            {%- else %}
                {{- raise_exception('Only text chunks are supported in system message contents.') }}
            {%- endif %}
        {%- endfor %}
    {%- endif %}
    {{- '[/SYSTEM_PROMPT]' -}}
    {%- set loop_messages = messages[1:] %}
{%- else %}
    {%- set loop_messages = messages %}
    {#- 移除 default_system_message 的依賴，簡化邏輯 -#}
{%- endif %}
{#- Tools definition #}
{%- set tools_definition = '' %}
{%- set has_tools = false %}
{%- if tools is defined and tools is not none and tools|length > 0 %}
    {%- set has_tools = true %}
    {%- set tools_definition = '[AVAILABLE_TOOLS]' + (tools| tojson) + '[/AVAILABLE_TOOLS]' %}
    {{- tools_definition }}
{%- endif %}
{#- Checks for alternating user/assistant messages. #}
{%- set ns = namespace(index=0) %}
{%- for message in loop_messages %}
    {%- if message.role == 'user' or (message.role == 'assistant' and (message.tool_calls is not defined or message.tool_calls is none or message.tool_calls | length == 0)) %}
        {%- if (message['role'] == 'user') != (ns.index % 2 == 0) %}
            {{- raise_exception('After the optional system message, conversation roles must alternate user and assistant roles except for tool calls and results.') }}
        {%- endif %}
        {%- set ns.index = ns.index + 1 %}
    {%- endif %}
{%- endfor %}
{#- Handle conversation messages. #}
{%- for message in loop_messages %}
    {%- if message['role'] == 'user' %}
        {%- if message['content'] is string %}
            {{- '[INST]' + message['content'] + '[/INST]' }}
        {%- elif message['content'] | length > 0 %}
            {{- '[INST]' }}
            {%- if message['content'] | length == 2 %}
                {%- set blocks = message['content'] | sort(attribute='type') %}
            {%- else %}
                {%- set blocks = message['content'] %}
            {%- endif %}
            {%- for block in blocks %}
                {%- if block['type'] == 'text' %}
                    {{- block['text'] }}
                {%- elif block['type'] in ['image', 'image_url'] %}
                    {{- '[IMG]' }}
                {%- else %}
                    {{- raise_exception('Only text, image and image_url chunks are supported in user message content.') }}
                {%- endif %}
            {%- endfor %}
            {{- '[/INST]' }}
        {%- else %}
            {{- raise_exception('User message must have a string or a list of chunks in content') }}
        {%- endif %}
    {%- elif message['role'] == 'assistant' %}
        {%- if (message['content'] is none or message['content'] == '' or message['content']|length == 0) and (message['tool_calls'] is not defined or message['tool_calls'] is none or message['tool_calls']|length == 0) %}
            {{- raise_exception('Assistant message must have a string or a list of chunks in content or a list of tool calls.') }}
        {%- endif %}
        {%- if message['content'] is string %}
            {{- message['content'] }}
        {%- elif message['content'] | length > 0 %}
            {%- for block in message['content'] %}
                {%- if block['type'] == 'text' %}
                    {{- block['text'] }}
                {%- else %}
                    {{- raise_exception('Only text chunks are supported in assistant message contents.') }}
                {%- endif %}
            {%- endfor %}
        {%- endif %}
        
        {%- if message['tool_calls'] is defined and message['tool_calls'] is not none and message['tool_calls']|length > 0 %}
            {%- for tool in message['tool_calls'] %}
                {%- set arguments = tool['function']['arguments'] %}
                {%- if arguments is not string %}
                    {%- set arguments = arguments|tojson|safe %}
                {%- elif arguments == '' %}
                    {%- set arguments = '{}' %}
                {%- endif %}
                {{- '[TOOL_CALLS]' + tool['function']['name'] + '[ARGS]' + arguments }}
            {%- endfor %}
        {%- endif %}
        {{- eos_token }}
    {%- elif message['role'] == 'tool' %}
        {{- '[TOOL_RESULTS]' + message['content']|string + '[/TOOL_RESULTS]' }}
    {%- else %}
        {{- raise_exception('Only user, assistant and tool roles are supported, got ' + message['role'] + '.') }}
    {%- endif %}
{%- endfor %}
"""

# 3️⃣ 只新增你自己的 task tokens
task_tokens = [
    "<|task-1-indicator|>",
    "<|task-2-indicator|>",
    "<|task-2-bow|>",
    "<|task-2-eow|>",
    "<|task-3-indicator|>",
    "<|task-4_1-indicator|>",
    "<|task-4_2-indicator|>",
]

added = tokenizer.add_special_tokens({"additional_special_tokens": task_tokens})
print(f"✅ 新增 {added} 個 task tokens")

# 4️⃣ 加上你的 chat template
tokenizer.chat_template = mistral_chat_template

# 5️⃣ 儲存
save_path = "/home/chris/LLM-Training/src/llm_training/tokenizer_mistral"
tokenizer.save_pretrained(save_path)
print(f"💾 已儲存至 {save_path}")


/home/chris/miniconda3/envs/llm-training/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 新增 7 個 task tokens
💾 已儲存至 /home/chris/LLM-Training/src/llm_training/tokenizer_mistral


In [1]:
import json
import os

# 定義路徑
base_path = "/home/chris/LLM-Training/src/llm_training/tokenizer_mistral"
config_path = os.path.join(base_path, "tokenizer_config.json")

print(f"📂 正在讀取設定檔: {config_path}")

# 1. 讀取 JSON
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

# ==========================================
# 🩹 修復 1: 將 extra_special_tokens 從 List 改為 Dict
# ==========================================
# 這是導致 transformers 崩潰的主因
if "extra_special_tokens" in config:
    est = config["extra_special_tokens"]
    if isinstance(est, list):
        print(f"🔧 發現 extra_special_tokens 為 List 格式 (長度 {len(est)})，正在轉換為 Dict...")
        # 將 ["token1", "token2"] 轉換為 {"token1": {}, "token2": {}}
        config["extra_special_tokens"] = {token: {} for token in est}
    else:
        print("✅ extra_special_tokens 格式正確 (Dict)。")

# ==========================================
# 📝 修復 2: 寫入安全的 Chat Template
# ==========================================
print("📝 正在更新 Chat Template...")

safe_mistral_template = """{%- set default_system_message = default_system_message | default('') -%}
{#- Begin of sequence token. #}
{{- bos_token }}
{#- Handle system prompt if it exists. #}
{%- if messages[0]['role'] == 'system' %}
    {{- '[SYSTEM_PROMPT]' -}}
    {%- if messages[0]['content'] is string %}
        {{- messages[0]['content'] -}}
    {%- else %}        
        {%- for block in messages[0]['content'] %}
            {%- if block['type'] == 'text' %}
                {{- block['text'] }}
            {%- else %}
                {{- raise_exception('Only text chunks are supported in system message contents.') }}
            {%- endif %}
        {%- endfor %}
    {%- endif %}
    {{- '[/SYSTEM_PROMPT]' -}}
    {%- set loop_messages = messages[1:] %}
{%- else %}
    {%- set loop_messages = messages %}
    {%- if default_system_message != '' %}
        {{- '[SYSTEM_PROMPT]' + default_system_message + '[/SYSTEM_PROMPT]' }}
    {%- endif %}
{%- endif %}
{#- Tools definition #}
{%- set tools_definition = '' %}
{%- set has_tools = false %}
{%- if tools is defined and tools is not none and tools|length > 0 %}
    {%- set has_tools = true %}
    {%- set tools_definition = '[AVAILABLE_TOOLS]' + (tools| tojson) + '[/AVAILABLE_TOOLS]' %}
    {{- tools_definition }}
{%- endif %}
{#- Checks for alternating user/assistant messages. #}
{%- set ns = namespace(index=0) %}
{%- for message in loop_messages %}
    {%- if message.role == 'user' or (message.role == 'assistant' and (message.tool_calls is not defined or message.tool_calls is none or message.tool_calls | length == 0)) %}
        {%- if (message['role'] == 'user') != (ns.index % 2 == 0) %}
            {{- raise_exception('After the optional system message, conversation roles must alternate user and assistant roles except for tool calls and results.') }}
        {%- endif %}
        {%- set ns.index = ns.index + 1 %}
    {%- endif %}
{%- endfor %}
{#- Handle conversation messages. #}
{%- for message in loop_messages %}
    {%- if message['role'] == 'user' %}
        {%- if message['content'] is string %}
            {{- '[INST]' + message['content'] + '[/INST]' }}
        {%- elif message['content'] | length > 0 %}
            {{- '[INST]' }}
            {%- if message['content'] | length == 2 %}
                {%- set blocks = message['content'] | sort(attribute='type') %}
            {%- else %}
                {%- set blocks = message['content'] %}
            {%- endif %}
            {%- for block in blocks %}
                {%- if block['type'] == 'text' %}
                    {{- block['text'] }}
                {%- elif block['type'] in ['image', 'image_url'] %}
                    {{- '[IMG]' }}
                {%- else %}
                    {{- raise_exception('Only text, image and image_url chunks are supported in user message content.') }}
                {%- endif %}
            {%- endfor %}
            {{- '[/INST]' }}
        {%- else %}
            {{- raise_exception('User message must have a string or a list of chunks in content') }}
        {%- endif %}
    {%- elif message['role'] == 'assistant' %}
        {%- if (message['content'] is none or message['content'] == '' or message['content']|length == 0) and (message['tool_calls'] is not defined or message['tool_calls'] is none or message['tool_calls']|length == 0) %}
            {{- raise_exception('Assistant message must have a string or a list of chunks in content or a list of tool calls.') }}
        {%- endif %}
        {%- if message['content'] is string %}
            {{- message['content'] }}
        {%- elif message['content'] | length > 0 %}
            {%- for block in message['content'] %}
                {%- if block['type'] == 'text' %}
                    {{- block['text'] }}
                {%- else %}
                    {{- raise_exception('Only text chunks are supported in assistant message contents.') }}
                {%- endif %}
            {%- endfor %}
        {%- endif %}
        
        {%- if message['tool_calls'] is defined and message['tool_calls'] is not none and message['tool_calls']|length > 0 %}
            {%- for tool in message['tool_calls'] %}
                {%- set arguments = tool['function']['arguments'] %}
                {%- if arguments is not string %}
                    {%- set arguments = arguments|tojson|safe %}
                {%- elif arguments == '' %}
                    {%- set arguments = '{}' %}
                {%- endif %}
                {{- '[TOOL_CALLS]' + tool['function']['name'] + '[ARGS]' + arguments }}
            {%- endfor %}
        {%- endif %}
        {{- eos_token }}
    {%- elif message['role'] == 'tool' %}
        {{- '[TOOL_RESULTS]' + message['content']|string + '[/TOOL_RESULTS]' }}
    {%- else %}
        {{- raise_exception('Only user, assistant and tool roles are supported, got ' + message['role'] + '.') }}
    {%- endif %}
{%- endfor %}
"""

config["chat_template"] = safe_mistral_template

# ==========================================
# 3. 儲存
# ==========================================
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f"✅ 設定檔修復完成！已儲存至: {config_path}")
print("🎉 您現在可以安全地進行資料預處理與訓練了 (transformers 不會再報錯了)。")

📂 正在讀取設定檔: /home/chris/LLM-Training/src/llm_training/tokenizer_mistral/tokenizer_config.json
✅ extra_special_tokens 格式正確 (Dict)。
📝 正在更新 Chat Template...
✅ 設定檔修復完成！已儲存至: /home/chris/LLM-Training/src/llm_training/tokenizer_mistral/tokenizer_config.json
🎉 您現在可以安全地進行資料預處理與訓練了 (transformers 不會再報錯了)。


## 放大選詞填空

In [1]:
import os
import json
import numpy as np
from datasets import load_dataset, load_from_disk, concatenate_datasets, Dataset

In [2]:
# ================= 1. 設定參數與路徑 =================
# Task 1~3 的資料來源
T123_DATASET_ID = "TAIDE-EDU/Edu-TAIDE-IT-Data"
T123_CONFIG = "task-1-2-3-241127"

# Task 4 的資料來源 (已合併好的本地資料)
T4_DATASET_PATH = "./processed_all_data"

# 最終輸出的路徑
OUTPUT_PATH = "./final_balanced_dataset"

# Special Tokens 定義 (請確認是否與您的資料一致)
# 假設 Task 1~3 的 token 命名邏輯與 Task 4 類似
TOKEN_MAP = {
    "task_1": "<|task-1-indicator|>",
    "task_2": "<|task-2-indicator|>",
    "task_3": "<|task-3-indicator|>",
    "task_4_1": "<|task-4_1-indicator|>", 
    "task_4_2": "<|task-4_2-indicator|>"
}

# (可選) 如果您後續程式碼需要檢查資料內容是否完整，可以另外定義變數
# 但不要放入上面的 TOKEN_MAP 迴圈中
TASK_2_TOKENS = {
    "bow": "<|task-2-bow|>",
    "eow": "<|task-2-eow|>"
}

In [3]:
# ================= 2. 載入資料集 =================
print("正在載入 Task 1~3 資料...")
try:
    ds_t123 = load_dataset(T123_DATASET_ID, T123_CONFIG, split="train")
    print(f" -> Task 1~3 原始筆數: {len(ds_t123)}")
except Exception as e:
    print(f"載入 Task 1~3 失敗: {e}")
    ds_t123 = None

print("\n正在載入 Task 4 資料...")
try:
    ds_t4 = load_from_disk(T4_DATASET_PATH)
    if 'train' in ds_t4: ds_t4 = ds_t4['train'] # 處理 DatasetDict
    print(f" -> Task 4 原始筆數: {len(ds_t4)}")
except Exception as e:
    print(f"載入 Task 4 失敗: {e}")
    ds_t4 = None

正在載入 Task 1~3 資料...
 -> Task 1~3 原始筆數: 146029

正在載入 Task 4 資料...
 -> Task 4 原始筆數: 11695


In [4]:
# ================= 3. 資料分類與清洗 =================

def is_valid_data(example):
    """檢查資料是否有效 (非空值)"""
    msgs = example.get('messages')
    if msgs is None or len(msgs) == 0:
        return False
    if 'content' not in msgs[0] or not msgs[0]['content']:
        return False
    return True

def filter_by_token(dataset, token):
    """根據 Token 篩選資料"""
    return dataset.filter(lambda x: token in x['messages'][0]['content'])

# === [新增] Task 2 完整性檢查函數 ===
def is_valid_task2(example):
    """確保 Task 2 除了有指示符外，還有 bow 和 eow"""
    content = example['messages'][0]['content']
    # 檢查是否同時包含 bow 和 eow
    has_bow = "<|task-2-bow|>" in content
    has_eow = "<|task-2-eow|>" in content
    return has_bow and has_eow

# 先合併兩個來源以便統一處理 (暫時)
# 注意：這裡假設兩個 dataset 的欄位 schema 已經對齊
raw_datasets = []
if ds_t123: raw_datasets.append(ds_t123)
if ds_t4: raw_datasets.append(ds_t4)

if not raw_datasets:
    raise ValueError("沒有載入任何資料！")

full_raw_ds = concatenate_datasets(raw_datasets)
print(f"\n資料合併後總筆數 (清洗前): {len(full_raw_ds)}")

# 清洗空資料
full_raw_ds = full_raw_ds.filter(is_valid_data)
print(f"資料清洗後總筆數: {len(full_raw_ds)}")

# 開始分類
categorized_ds = {}
print("\n正在進行 Task 分類...")

for task_name, token in TOKEN_MAP.items():
    print(f" -> 正在篩選 {task_name}...")
    subset = filter_by_token(full_raw_ds, token)
    
    # === [新增] 針對 Task 2 的特殊過濾 ===
    if task_name == "task_2":
        original_count = len(subset)
        subset = subset.filter(is_valid_task2)
        filtered_count = len(subset)
        if original_count != filtered_count:
            print(f"    [警告] 發現 {original_count - filtered_count} 筆 Task 2 資料結構不完整 (缺少 bow/eow)，已剔除。")
    # ====================================

    categorized_ds[task_name] = subset
    print(f"    {task_name}: {len(subset)} 筆")


資料合併後總筆數 (清洗前): 157724


Filter: 100%|██████████| 157724/157724 [00:01<00:00, 105025.94 examples/s]


資料清洗後總筆數: 156520

正在進行 Task 分類...
 -> 正在篩選 task_1...


Filter: 100%|██████████| 156520/156520 [00:01<00:00, 103090.45 examples/s]


    task_1: 33361 筆
 -> 正在篩選 task_2...


...

    task_3: 8740 筆
 -> 正在篩選 task_4_1...


Filter: 100%|██████████| 156520/156520 [00:01<00:00, 103591.53 examples/s]


    task_4_1: 5893 筆
 -> 正在篩選 task_4_2...


Filter: 100%|██████████| 156520/156520 [00:01<00:00, 103688.16 examples/s]

    task_4_2: 4598 筆


In [5]:
# ================= 4. 執行平衡採樣 (保守策略) =================
# 
# 策略說明：
# - Task 4-1 (閱讀) 和 Task 4-2 (填空) 放大到 15,000 筆 (約 2.5-3.3x)
# - Task 2 (詞表) 下採樣到 50,000 筆，避免過度主導
# - Task 1, Task 3 維持原樣
#
# 這樣可以：
# 1. 讓模型區分「閱讀測驗」和「選詞填空」
# 2. 避免過度放大導致過擬合
# 3. 減少詞表任務的佔比

TARGET_TASK4 = 15000  # Task 4-1, 4-2 的目標數量
TARGET_TASK2 = 50000  # Task 2 下採樣目標

print(f"\n=== 平衡採樣設定 ===")
print(f"Task 4-1/4-2 目標: {TARGET_TASK4} 筆")
print(f"Task 2 目標: {TARGET_TASK2} 筆")

final_datasets_list = []

for task_name, ds in categorized_ds.items():
    current_len = len(ds)
    
    if task_name in ['task_4_1', 'task_4_2']:
        # 🔥 放大閱讀測驗和選詞填空，但控制倍數
        if current_len < TARGET_TASK4:
            ratio = TARGET_TASK4 / current_len
            print(f"⬆️ 上採樣 {task_name}: {current_len:,} -> {TARGET_TASK4:,} ({ratio:.1f}x)")
            new_indices = np.random.choice(current_len, size=TARGET_TASK4, replace=True)
            final_datasets_list.append(ds.select(new_indices))
        else:
            print(f"✅ 維持 {task_name}: {current_len:,}")
            final_datasets_list.append(ds)
            
    elif task_name == 'task_2':
        # 🔥 詞表任務太多，下採樣
        if current_len > TARGET_TASK2:
            print(f"⬇️ 下採樣 {task_name}: {current_len:,} -> {TARGET_TASK2:,}")
            new_indices = np.random.choice(current_len, size=TARGET_TASK2, replace=False)
            final_datasets_list.append(ds.select(new_indices))
        else:
            print(f"✅ 維持 {task_name}: {current_len:,}")
            final_datasets_list.append(ds)
    else:
        # Task 1, Task 3 維持原樣
        print(f"✅ 維持 {task_name}: {current_len:,}")
        final_datasets_list.append(ds)

print(f"\n處理完成，共 {len(final_datasets_list)} 個子資料集")


=== 平衡採樣設定 ===
Task 4-1/4-2 目標: 15000 筆
Task 2 目標: 50000 筆
✅ 維持 task_1: 33,361
⬇️ 下採樣 task_2: 103,928 -> 50,000
✅ 維持 task_3: 8,740
⬆️ 上採樣 task_4_1: 5,893 -> 15,000 (2.5x)
⬆️ 上採樣 task_4_2: 4,598 -> 15,000 (3.3x)

處理完成，共 5 個子資料集


In [6]:
# ================= 5. 最終合併與儲存 =================

print("\n正在合併最終資料集...")
final_dataset = concatenate_datasets(final_datasets_list)

# 打亂順序 (Shuffle) - 非常重要！
final_dataset = final_dataset.shuffle(seed=42)

print(f"🎉 最終資料集總筆數: {len(final_dataset)}")

print(f"正在儲存至 {OUTPUT_PATH} ...")
final_dataset.save_to_disk(OUTPUT_PATH)
print("完成！")


正在合併最終資料集...
🎉 最終資料集總筆數: 122101
正在儲存至 ./final_balanced_dataset ...


Saving the dataset (1/1 shards): 100%|██████████| 122101/122101 [00:00<00:00, 249540.86 examples/s]

完成！


In [7]:
ALL_DATA = "./final_balanced_dataset"
print("\n正在載入 新資料...")
try:
    ds_t5 = load_from_disk(ALL_DATA)
    if 'train' in ds_t5: ds_t5 = ds_t5['train'] # 處理 DatasetDict
    print(f" -> 新原始筆數: {len(ds_t5)}")
except Exception as e:
    print(f"載入新資料失敗: {e}")
    ds_t5 = None


正在載入 新資料...
 -> 新原始筆數: 122101


In [9]:
# ================= 3. 資料分類與清洗 =================

def is_valid_data(example):
    """檢查資料是否有效 (非空值)"""
    msgs = example.get('messages')
    if msgs is None or len(msgs) == 0:
        return False
    if 'content' not in msgs[0] or not msgs[0]['content']:
        return False
    return True

def filter_by_token(dataset, token):
    """根據 Token 篩選資料"""
    return dataset.filter(lambda x: token in x['messages'][0]['content'])

# 先合併兩個來源以便統一處理 (暫時)
# 注意：這裡假設兩個 dataset 的欄位 schema 已經對齊
raw_datasets = []
if ds_t5: raw_datasets.append(ds_t5)

if not raw_datasets:
    raise ValueError("沒有載入任何資料！")

full_raw_ds = concatenate_datasets(raw_datasets)
print(f"\n資料合併後總筆數 (清洗前): {len(full_raw_ds)}")

# 清洗空資料
full_raw_ds = full_raw_ds.filter(is_valid_data)
print(f"資料清洗後總筆數: {len(full_raw_ds)}")

# 開始分類
categorized_ds = {}
print("\n正在進行 Task 分類...")

for task_name, token in TOKEN_MAP.items():
    print(f" -> 正在篩選 {task_name}...")
    subset = filter_by_token(full_raw_ds, token)
    categorized_ds[task_name] = subset
    print(f"    {task_name}: {len(subset)} 筆")


資料合併後總筆數 (清洗前): 212751


Filter: 100%|██████████| 212751/212751 [00:02<00:00, 94841.84 examples/s]


資料清洗後總筆數: 212751

正在進行 Task 分類...
 -> 正在篩選 task_1...


Filter: 100%|██████████| 212751/212751 [00:02<00:00, 78184.84 examples/s]


    task_1: 33361 筆
 -> 正在篩選 task_2...


...

    task_3: 8740 筆
 -> 正在篩選 task_4_1...


Filter: 100%|██████████| 212751/212751 [00:02<00:00, 78432.12 examples/s]


    task_4_1: 33361 筆
 -> 正在篩選 task_4_2...


Filter: 100%|██████████| 212751/212751 [00:02<00:00, 81018.80 examples/s]

    task_4_2: 33361 筆


In [1]:
from datasets import load_from_disk, DatasetDict
import shutil
import os

# 1. 設定路徑
current_path = "/home/chris/LLM-Training/final_balanced_dataset"
new_path = "/home/chris/LLM-Training/final_balanced_dataset_fixed"

print(f"正在讀取資料: {current_path}")
try:
    # 載入資料
    dataset = load_from_disk(current_path)
    print(f"資料類型: {type(dataset)}")

    # 2. 檢查並修復
    if not isinstance(dataset, DatasetDict):
        print("偵測到單一 Dataset，正在轉換為 DatasetDict...")
        
        # 將單一 Dataset 包裝進 'train' split 中
        # 這樣程式碼做 dataset['train'] 時就能拿到資料了
        fixed_dataset = DatasetDict({"train": dataset})
        
        # 3. 儲存到新路徑
        print(f"正在儲存修復後的資料至: {new_path}")
        fixed_dataset.save_to_disk(new_path)
        print("✅ 儲存完成！")
        
    else:
        print("資料已經是 DatasetDict，請檢查鍵值:", dataset.keys())
        # 如果已經是 Dict 但沒有 train (例如只有 'train_set')，這也是問題
        if 'train' not in dataset.keys():
            print("❌ 錯誤: DatasetDict 中缺少 'train' 鍵值。")
            # 如果您的鍵值叫別的 (例如 'data')，請手動改名
            # fixed_dataset = DatasetDict({"train": dataset['data']})
            # fixed_dataset.save_to_disk(new_path)

except Exception as e:
    print(f"發生錯誤: {e}")

正在讀取資料: /home/chris/LLM-Training/final_balanced_dataset
資料類型: <class 'datasets.arrow_dataset.Dataset'>
偵測到單一 Dataset，正在轉換為 DatasetDict...
正在儲存修復後的資料至: /home/chris/LLM-Training/final_balanced_dataset_fixed


Saving the dataset (2/2 shards): 100%|██████████| 212751/212751 [00:00<00:00, 733756.95 examples/s]

✅ 儲存完成！


In [2]:
from datasets import load_from_disk
from transformers import PreTrainedTokenizerFast

# 1. 載入剛重新處理好的資料
dataset_path = "/home/chris/LLM-Training/final_balanced_dataset_fixed"
try:
    ds = load_from_disk(dataset_path)
    print("資料載入成功！")
    
    # 2. 載入 Tokenizer
    tokenizer_path = "/home/chris/LLM-Training/src/llm_training/tokenizer_mistral"
    tokenizer = PreTrainedTokenizerFast.from_pretrained(tokenizer_path)
    
    # 3. 抽樣檢查 (解碼第一筆資料)
    sample_ids = ds['train'][0]['input_ids']
    print(f"Sample IDs (前20個): {sample_ids[:20]}")
    
    decoded_text = tokenizer.decode(sample_ids, skip_special_tokens=False)
    print("\n========= 解碼結果 =========")
    print(decoded_text[:500]) # 印出前500字
    print("===========================")
    
    if "The capital of France" in decoded_text or "SYSTEM_PROMPT" in decoded_text:
        print("✅ 恭喜！資料是正常的！")
    else:
        print("❌ 警告！解碼出來還是亂碼，請不要開始訓練！")
        
except Exception as e:
    print(f"還沒處理好資料，請先執行預處理。錯誤: {e}")

資料載入成功！
還沒處理好資料，請先執行預處理。錯誤: 'input_ids'


In [3]:
from datasets import load_from_disk
from transformers import PreTrainedTokenizerFast

dataset_path = "/home/chris/LLM-Training/final_balanced_dataset_fixed"
ds = load_from_disk(dataset_path)
print("type(ds):", type(ds))

# 1) 看有哪些 split / 或單一 dataset
if hasattr(ds, "keys"):
    print("splits:", list(ds.keys()))
    first_split = list(ds.keys())[0]
    d = ds[first_split]
else:
    print("single dataset (no splits)")
    d = ds

print("num_rows:", len(d))
print("column_names:", d.column_names)

# 2) 看第一筆實際長什麼樣
ex0 = d[0]
print("example[0] keys:", list(ex0.keys()))

# 3) 載入 tokenizer
tokenizer_path = "/home/chris/LLM-Training/src/llm_training/tokenizer_mistral"
tokenizer = PreTrainedTokenizerFast.from_pretrained(tokenizer_path)

# 4) 找出可能是 token ids 的欄位並嘗試解碼
candidate_fields = [k for k,v in ex0.items()
                    if isinstance(v, (list, tuple)) and len(v) > 0 and isinstance(v[0], int)]
print("candidate token fields:", candidate_fields)

for k in candidate_fields:
    ids = ex0[k]
    print(f"\n--- decode field: {k} (len={len(ids)}) ---")
    print(tokenizer.decode(ids[:200], skip_special_tokens=False))
    break

# 5) 若是文字欄位，也列出前 500 字
text_fields = [k for k,v in ex0.items() if isinstance(v, str)]
print("\ntext fields:", text_fields)
for k in text_fields[:3]:
    print(f"\n--- text field: {k} ---")
    print(ex0[k][:500])


type(ds): <class 'datasets.dataset_dict.DatasetDict'>
splits: ['train']
num_rows: 212751
column_names: ['messages', 'source', 'metadata']
example[0] keys: ['messages', 'source', 'metadata']
candidate token fields: []

text fields: ['source', 'metadata']

--- text field: source ---
task-2

--- text field: metadata ---
{"ans": "都、皆、全。", "classical": 0, "explanations": ["都、皆、全。", "普及、遍及。", "和睦。", "《易經》卦名。六十四卦之一。艮（☶）下兌（☱）上。表示山澤相感應之象。", "姓。"], "position": [2, 3], "sentence": "老少咸宜。", "word": "咸"}


In [12]:
import json
import os

# 設定模型路徑
model_dir = "/home/chris/Ministral-3-14B-BF16-Fixed"
config_path = os.path.join(model_dir, "config.json")

print(f"正在讀取: {config_path}")

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

# 檢查是否有 text_config (這是多模態模型的特徵)
if "text_config" in config:
    print("偵測到多模態巢狀結構 (Multimodal Nested Config)。")
    print("正在執行「扁平化」手術，提取文字模型設定...")
    
    # 1. 提取 text_config 的內容
    text_config = config["text_config"]
    
    # 2. 將 text_config 的內容覆蓋到最外層
    # 這會把 hidden_size, num_hidden_layers 等關鍵參數拉到最上面
    for key, value in text_config.items():
        config[key] = value
    
    # 3. 確保 model_type 是 mistral
    config["model_type"] = "mistral"
    
    # 4. 確保架構名稱是 CausalLM (否則可能預設是 Model，不會載入 LM Head)
    config["architectures"] = ["MistralForCausalLM"]
    
    # 5. 清理多模態相關的欄位 (避免干擾)
    keys_to_remove = [
        "text_config", 
        "vision_config", 
        "vision_feature_layer", 
        "image_token_index",
        "multimodal_projector_bias",
        "projector_hidden_act",
        "spatial_merge_size"
    ]
    
    for k in keys_to_remove:
        if k in config:
            del config[k]
            
    # 6. 再次清理 generation_config (以防萬一)
    if "generation_config" in config:
        del config["generation_config"]

    # 寫回檔案
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)
        
    print("✅ 手術成功！Config 已轉換為標準 Mistral 純文字格式。")
    print("現在 Keys:", list(config.keys()))
    
else:
    print("未發現 text_config，無需扁平化。")
    # 如果沒有 text_config 但之前有 generation_config 問題，腳本可能已經處理過了

正在讀取: /home/chris/Ministral-3-14B-BF16-Fixed/config.json
未發現 text_config，無需扁平化。


In [2]:
from transformers import AutoTokenizer
from datasets import load_from_disk
import torch

# 1. 載入剛剛存好的 Tokenizer
tokenizer = AutoTokenizer.from_pretrained("/home/chris/LLM-Training/src/llm_training/tokenizer_mistral")

# 2. 載入資料集
dataset = load_from_disk("/home/chris/LLM-Training/final_balanced_dataset_fixed")
print("資料集載入成功，第一筆資料長這樣：")
sample_data = dataset['train'][0] # 假設有 train split
print(sample_data)

# 3. 測試 Template 應用
# 假設您的資料集欄位叫 'messages' 或 'conversations'
key_name = 'messages' if 'messages' in sample_data else 'conversations'
try:
    msgs = sample_data[key_name]
    
    # === [修正重點] 必須補上 default_system_message ===
    text = tokenizer.apply_chat_template(
        msgs, 
        tokenize=False, 
        default_system_message=""  # <-- 加上這行，給它一個空字串
    )
    # ===============================================
    
    print("\n[Template 轉換結果]:")
    print(text[:500]) 
    
    if "[INST]" in text:
        print("\n✅ 成功看到 [INST]，Template 運作正常！")
    else:
        print("\n❌ 沒看到 [INST]，Template 可能沒吃到資料！")
        
except Exception as e:
    print(f"\n❌ Template 應用失敗: {e}")
    print("請檢查資料集的欄位名稱是否跟 Jinja2 模板裡的 message['role'] 一致")

資料集載入成功，第一筆資料長這樣：
{'messages': [{'role': 'user', 'content': '<|task-2-indicator|>請判斷「咸」在以下句子中為何種解釋，並直接輸出正確的選項代號。\n老少<|task-2-bow|>咸<|task-2-eow|>宜。\n\nA. 都、皆、全。\nB. 普及、遍及。\nC. 和睦。\nD. 《易經》卦名。六十四卦之一。艮（☶）下兌（☱）上。表示山澤相感應之象。\nE. 姓。'}, {'role': 'assistant', 'content': 'A'}], 'source': 'task-2', 'metadata': '{"ans": "都、皆、全。", "classical": 0, "explanations": ["都、皆、全。", "普及、遍及。", "和睦。", "《易經》卦名。六十四卦之一。艮（☶）下兌（☱）上。表示山澤相感應之象。", "姓。"], "position": [2, 3], "sentence": "老少咸宜。", "word": "咸"}'}

[Template 轉換結果]:
<s>[INST]<|task-2-indicator|>請判斷「咸」在以下句子中為何種解釋，並直接輸出正確的選項代號。
老少<|task-2-bow|>咸<|task-2-eow|>宜。

A. 都、皆、全。
B. 普及、遍及。
C. 和睦。
D. 《易經》卦名。六十四卦之一。艮（☶）下兌（☱）上。表示山澤相感應之象。
E. 姓。[/INST]A</s>

✅ 成功看到 [INST]，Template 運作正常！


In [3]:
import torch
from transformers import AutoTokenizer
from datasets import load_from_disk

# 1. 設定路徑
data_path = "/home/chris/LLM-Training/final_balanced_dataset_fixed"
tokenizer_path = "/home/chris/LLM-Training/src/llm_training/tokenizer_mistral"

print("載入 Tokenizer 與 資料集...")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
dataset = load_from_disk(data_path)

# 2. 模擬一筆資料的處理
# 注意：這裡我們要模擬您的 DataModule 在做的事情
# 假設您的 DataModule 是用 apply_chat_template 轉成 text 後直接 tokenize

def check_one_sample(sample):
    # 這裡假設您的資料集欄位是 'messages' 或 'conversations'
    key = 'messages' if 'messages' in sample else 'conversations'
    msgs = sample[key]
    
    # A. 套用 Template
    text = tokenizer.apply_chat_template(
        msgs, 
        tokenize=False, 
        default_system_message=""  # <-- 加上這行，給它一個空字串
    )
    print(f"\n[原始文字片段]: {text[:100]} ...")
    
    # B. Tokenize (轉成 ID)
    input_ids = tokenizer(text, return_tensors="pt", padding=False, add_special_tokens=False).input_ids[0]
    
    print(f"[Token 數量]: {len(input_ids)}")
    
    # C. 模擬 Masking (這是最關鍵的一步！)
    # 我們嘗試找找看 tokenizer 是否認得 [/INST]
    # Mistral 官方通常用 ID 找，或是用文字找
    
    # 測試 1: 看看 input_ids 裡有沒有 [/INST] 的 ID
    # 先找出 [/INST] 對應的 ID 是什麼
    # 注意：有時候是單個 token，有時候是多個
    inst_end_token = "[/INST]"
    inst_end_id = tokenizer.convert_tokens_to_ids(inst_end_token)
    
    print(f"Token '{inst_end_token}' 的 ID 是: {inst_end_id}")
    
    if inst_end_id in input_ids:
        print(f"✅ 在 input_ids 中找到了 [/INST] (ID: {inst_end_id})")
        # 顯示該 ID 出現的位置
        locs = (input_ids == inst_end_id).nonzero(as_tuple=True)[0]
        print(f"   出現位置索引: {locs}")
    else:
        print(f"❌ 在 input_ids 中找不到 [/INST] (ID: {inst_end_id})")
        print("   這就是 Loss=0 的原因！Masking 邏輯找不到分隔點，把整句都蓋掉了。")
        
        # 加碼檢查：它是不是被切碎了？
        print("   檢查是否被切碎...")
        tokens = tokenizer.convert_ids_to_tokens(input_ids)
        # 印出包含 INST 附近的 tokens
        for i, t in enumerate(tokens):
            if "INST" in str(t):
                print(f"   發現碎裂的 Token: '{t}' at index {i}")

# 執行檢查
check_one_sample(dataset['train'][0])

載入 Tokenizer 與 資料集...

[原始文字片段]: <s>[INST]<|task-2-indicator|>請判斷「咸」在以下句子中為何種解釋，並直接輸出正確的選項代號。
老少<|task-2-bow|>咸<|task-2-eow|>宜。

A. 都 ...
[Token 數量]: 120
Token '[/INST]' 的 ID 是: 4
✅ 在 input_ids 中找到了 [/INST] (ID: 4)
   出現位置索引: tensor([117])


In [4]:
from datasets import load_from_disk

# 修改成您目前 YAML 裡填寫的路徑
current_path = "/home/chris/LLM-Training/final_balanced_dataset_fixed" 

try:
    ds = load_from_disk(current_path)
    print(f"資料集路徑: {current_path}")
    print(f"包含的欄位 (Features): {ds['train'].column_names}")
    
    if 'messages' in ds['train'].column_names:
        print("✅ 好消息！'messages' 欄位還在，這應該是原始資料。")
    else:
        print("❌ 壞消息！'messages' 欄位不見了，這是已經被處理過的資料。")
        print("⚠️ 您不能拿這個資料來重新跑 pre-process，因為沒有原始文字了！")
except Exception as e:
    print(f"讀取錯誤: {e}")

資料集路徑: /home/chris/LLM-Training/final_balanced_dataset_fixed
包含的欄位 (Features): ['messages', 'source', 'metadata']
✅ 好消息！'messages' 欄位還在，這應該是原始資料。


In [1]:
import transformers.tokenization_utils_base as tub
from transformers import PreTrainedTokenizerFast

# ==========================================
# 🩹 Monkey Patch: 動態修正 transformers 的 Bug
# ==========================================
# 備份原始方法
original_method = tub.PreTrainedTokenizerBase._set_model_specific_special_tokens

def patched_method(self, special_tokens=None):
    # 如果遇到 List 格式 (造成報錯的主因)，就將其視為空字典忽略
    if isinstance(special_tokens, list):
        print(f"⚠️ [Patch] 偵測到 extra_special_tokens 為 List 格式，已自動修正為 Dict 以避免崩潰。")
        special_tokens = {} 
    return original_method(self, special_tokens)

# 套用修正
tub.PreTrainedTokenizerBase._set_model_specific_special_tokens = patched_method
# ==========================================

# 設定路徑
SRC = "mistralai/Ministral-3-14B-Instruct-2512"
DST = "/home/chris/Ministral-3-14B-BF16-Fixed"

print(f"⬇️ 正在下載官方 Tokenizer: {SRC}")

try:
    # 使用 PreTrainedTokenizerFast 來載入 (它對 JSON 格式最友善)
    tokenizer = PreTrainedTokenizerFast.from_pretrained(
        SRC, 
        trust_remote_code=True
    )
    
    print(f"💾 正在儲存修復後的 Tokenizer 到: {DST}...")
    tokenizer.save_pretrained(DST)
    
    print("✅ Tokenizer 修復成功！")

except Exception as e:
    print(f"❌ 修復失敗: {e}")

/home/chris/miniconda3/envs/llm-training/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⬇️ 正在下載官方 Tokenizer: mistralai/Ministral-3-14B-Instruct-2512


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'TokenizersBackend'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


⚠️ [Patch] 偵測到 extra_special_tokens 為 List 格式，已自動修正為 Dict 以避免崩潰。


The tokenizer you are loading from 'mistralai/Ministral-3-14B-Instruct-2512' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


💾 正在儲存修復後的 Tokenizer 到: /home/chris/Ministral-3-14B-BF16-Fixed...
✅ Tokenizer 修復成功！


In [1]:
import transformers
print(f"Transformers Version: {transformers.__version__}")

/home/chris/miniconda3/envs/llm-training/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers Version: 4.57.3


In [17]:
! pip install -U transformers tokenizers huggingface_hub

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached huggingface_hub-1.2.3-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 31.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 35.3 MB/s  0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.20.3
    Uninstalling tokenizers-0.20.3:
      Successfully uninstalled tokenizers-0.20.3
  Attempting uninstall: transformers
    Found existing installation: transformers 4.46.3
    Uninstalling transformers-4.46.3:
      Successfully uninstalled transformers-4.46.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llm-training 0.2.0 requires tokenizers==0.20.3, but you have tokenizers 0.22.1 which is incompatible.
llm-training 0.2.0 requires transformers==4.46.3, but you have

In [1]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="mistralai/Ministral-3-14B-Instruct-2512-BF16",
    local_dir="/home/chris/Ministral-3-14B-BF16-Fixed",
    local_dir_use_symlinks=False
)


/home/chris/miniconda3/envs/llm-training/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/chris/miniconda3/envs/llm-training/lib/python3.10/site-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 20 files: 100%|██████████| 20/20 [07:02<00:00, 21.13s/it]


'/home/chris/Ministral-3-14B-BF16-Fixed'

In [ ]:
import torch
from transformers import AutoProcessor, Mistral3ForConditionalGeneration

model_path = "/home/chris/Ministral-3-14B-BF16-Fixed"

processor = AutoProcessor.from_pretrained(
    model_path,
    trust_remote_code=True,
    fix_mistral_regex=True,   # 先保留，下面會說怎麼讓它不需要
)

model = Mistral3ForConditionalGeneration.from_pretrained(
    model_path,
    device_map="auto",
    dtype=torch.bfloat16,
    trust_remote_code=True
)

prompt = "The capital of France is"
inputs = processor(text=prompt, return_tensors="pt").to(model.device)

out = model.generate(**inputs, max_new_tokens=20, do_sample=False)
print(processor.decode(out[0], skip_special_tokens=True))


Loading checkpoint shards: 100%|██████████| 6/6 [00:02<00:00,  2.87it/s]


The capital of France is Paris. Paris is a city that is known for its art, fashion, gastronomy, and culture


: 

In [3]:
#!/usr/bin/env python3
"""
修復 Ministral-3 tokenizer 的 chat template (v2)
修正 Jinja2 語法問題
"""

import json
import shutil
from pathlib import Path
from transformers import AutoTokenizer

# 修改為你的路徑
TOKENIZER_PATH = "/home/chris/LLM-Training/src/llm_training/tokenizer_mistral"
BACKUP_PATH = "/home/chris/LLM-Training/src/llm_training/tokenizer_mistral_backup"

def main():
    print("="*70)
    print("🔧 修復 Ministral-3 Chat Template (v2)")
    print("="*70)
    
    # 1. 備份原始 tokenizer
    tokenizer_path = Path(TOKENIZER_PATH)
    backup_path = Path(BACKUP_PATH)
    
    if not backup_path.exists():
        print(f"\n📦 備份原始 tokenizer 到: {backup_path}")
        shutil.copytree(tokenizer_path, backup_path)
        print("✅ 備份完成")
    else:
        print(f"\n📦 備份已存在: {backup_path}")
    
    # 2. 載入 tokenizer
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, trust_remote_code=True)
    
    # 3. 顯示原始 chat template
    print(f"\n{'='*50}")
    print("📌 原始 Chat Template (部分):")
    print(f"{'='*50}")
    original_template = tokenizer.chat_template
    print(original_template[:500] + "...")
    
    # 4. 修改原始 template，在 assistant 部分加入 {% generation %}
    # 關鍵是要在原始 template 的基礎上修改，而不是完全替換
    
    # 找到 assistant message 處理的部分並添加 generation 標記
    new_template = """{%- set default_system_message = default_system_message | default('') -%}
{{- bos_token }}
{%- if messages[0]['role'] == 'system' %}
    {{- '[SYSTEM_PROMPT]' + messages[0]['content'] + '[/SYSTEM_PROMPT]' }}
    {%- set loop_messages = messages[1:] %}
{%- else %}
    {%- set loop_messages = messages %}
    {%- if default_system_message != '' %}
        {{- '[SYSTEM_PROMPT]' + default_system_message + '[/SYSTEM_PROMPT]' }}
    {%- endif %}
{%- endif %}
{%- for message in loop_messages %}
    {%- if message['role'] == 'user' %}
        {{- '[INST]' + message['content'] + '[/INST]' }}
    {%- elif message['role'] == 'assistant' %}
{% generation %}{{ message['content'] }}{{ eos_token }}{% endgeneration %}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
{% generation %}{% endgeneration %}
{%- endif %}"""

    print(f"\n{'='*50}")
    print("📌 新的 Chat Template:")
    print(f"{'='*50}")
    print(new_template)
    
    # 5. 測試新 template
    print(f"\n{'='*50}")
    print("🧪 測試新 Template...")
    print(f"{'='*50}")
    
    tokenizer.chat_template = new_template
    
    messages = [
        {"role": "user", "content": "Hello"},
        {"role": "assistant", "content": "Hi there!"}
    ]
    
    try:
        result = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_assistant_tokens_mask=True,
            add_generation_prompt=False
        )
        
        input_ids = result['input_ids']
        assistant_mask = result.get('assistant_masks', [])
        
        print(f"\n✅ Template 測試成功!")
        print(f"  Input IDs 長度: {len(input_ids)}")
        print(f"  Assistant tokens 數量: {sum(assistant_mask)}")
        
        print(f"\n📌 逐 Token 分析:")
        for i, (tid, mask) in enumerate(zip(input_ids, assistant_mask)):
            token_str = tokenizer.decode([tid]).replace('\n', '\\n')
            label = "🟢 TRAIN" if mask else "⬜ SKIP"
            print(f"  [{i:3d}] ID={tid:6d} | {label} | '{token_str}'")
        
        if sum(assistant_mask) > 0:
            print(f"\n✅ 成功！Assistant tokens 會被正確標記！")
            
            # 保存
            save = input("\n是否保存修改後的 tokenizer? (y/n): ").strip().lower()
            if save == 'y':
                tokenizer.save_pretrained(TOKENIZER_PATH)
                print(f"✅ 已保存到: {TOKENIZER_PATH}")
            else:
                print("⚠️ 未保存，tokenizer 維持原狀")
        else:
            print(f"\n❌ assistant_mask 仍然全是 0，嘗試其他方案...")
            try_alternative_templates(tokenizer, messages)
            
    except Exception as e:
        print(f"\n❌ Template 測試失敗: {e}")
        import traceback
        traceback.print_exc()
        
        print("\n🔄 嘗試其他 template 方案...")
        try_alternative_templates(tokenizer, messages)


def try_alternative_templates(tokenizer, messages):
    """嘗試多種不同的 template 格式"""
    
    templates = {
        "v1_simple": """{{ bos_token }}{% for message in messages %}{% if message['role'] == 'user' %}[INST]{{ message['content'] }}[/INST]{% elif message['role'] == 'assistant' %}{% generation %}{{ message['content'] }}{{ eos_token }}{% endgeneration %}{% endif %}{% endfor %}""",
        
        "v2_multiline": """{%- for message in messages -%}
{%- if message['role'] == 'user' -%}
{{ bos_token }}[INST]{{ message['content'] }}[/INST]
{%- elif message['role'] == 'assistant' -%}
{% generation %}{{ message['content'] }}{{ eos_token }}{% endgeneration %}
{%- endif -%}
{%- endfor -%}""",

        "v3_no_bos_in_loop": """{{ bos_token }}{% for message in messages %}{% if message['role'] == 'user' %}[INST]{{ message['content'] }}[/INST]{% elif message['role'] == 'assistant' %}{% generation %}{{ message['content'] }}{% endgeneration %}{{ eos_token }}{% endif %}{% endfor %}""",

        "v4_eos_inside": """{{ bos_token }}{% for message in messages %}{% if message['role'] == 'user' %}[INST]{{ message['content'] }}[/INST]{% elif message['role'] == 'assistant' %}{% generation %}{{ message['content'] }}{{ eos_token }}{% endgeneration %}{% endif %}{% endfor %}{% if add_generation_prompt %}{% generation %}{% endgeneration %}{% endif %}""",
    }
    
    original_template = tokenizer.chat_template
    
    for name, template in templates.items():
        print(f"\n{'='*50}")
        print(f"🧪 嘗試 {name}...")
        print(f"{'='*50}")
        
        try:
            tokenizer.chat_template = template
            
            result = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                return_dict=True,
                return_assistant_tokens_mask=True,
                add_generation_prompt=False
            )
            
            input_ids = result['input_ids']
            assistant_mask = result.get('assistant_masks', [])
            
            print(f"  ✅ 語法正確")
            print(f"  Input IDs: {len(input_ids)}")
            print(f"  Assistant tokens: {sum(assistant_mask)}")
            
            if sum(assistant_mask) > 0:
                print(f"\n  🎉 成功！這個 template 可以用！")
                print(f"\n📌 逐 Token 分析:")
                for i, (tid, mask) in enumerate(zip(input_ids, assistant_mask)):
                    token_str = tokenizer.decode([tid]).replace('\n', '\\n')
                    label = "🟢 TRAIN" if mask else "⬜ SKIP"
                    print(f"    [{i:3d}] ID={tid:6d} | {label} | '{token_str}'")
                
                save = input(f"\n是否使用 {name} 並保存? (y/n): ").strip().lower()
                if save == 'y':
                    tokenizer.save_pretrained(TOKENIZER_PATH)
                    print(f"✅ 已保存!")
                    return
            else:
                print(f"  ⚠️ assistant_mask 仍然是 0")
                
        except Exception as e:
            print(f"  ❌ 失敗: {e}")
    
    # 恢復原始
    tokenizer.chat_template = original_template
    print("\n⚠️ 所有 template 都失敗，已恢復原始設定")
    print("\n💡 建議：不使用 return_assistant_tokens_mask，改用手動標記 labels")


if __name__ == "__main__":
    main()

🔧 修復 Ministral-3 Chat Template (v2)

📦 備份已存在: /home/chris/LLM-Training/src/llm_training/tokenizer_mistral_backup

📌 原始 Chat Template (部分):
{%- set default_system_message = default_system_message | default('') -%}
{#- Begin of sequence token. #}
{{- bos_token }}
{#- Handle system prompt if it exists. #}
{%- if messages[0]['role'] == 'system' %}
    {{- '[SYSTEM_PROMPT]' -}}
    {%- if messages[0]['content'] is string %}
        {{- messages[0]['content'] -}}
    {%- else %}        
        {%- for block in messages[0]['content'] %}
            {%- if block['type'] == 'text' %}
                {{- block['text'] }}
            {%- els...

📌 新的 Chat Template:
{%- set default_system_message = default_system_message | default('') -%}
{{- bos_token }}
{%- if messages[0]['role'] == 'system' %}
    {{- '[SYSTEM_PROMPT]' + messages[0]['content'] + '[/SYSTEM_PROMPT]' }}
    {%- set loop_messages = messages[1:] %}
{%- else %}
    {%- set loop_messages = messages %}
    {%- if default_system_messag